# mine-tuning-model (RTX 5060 Ti 최적화 버전)

## 사전 준비 (터미널에서 먼저 실행)

```bash
# 1. .env 파일에 HF 토큰 설정
cp docker/.env.example docker/.env
# docker/.env 열고 HF_TOKEN 입력

# 2. SGLang 서버 실행
cd docker
docker compose up -d

# 3. 서버 로그 확인 (선택)
docker logs -f sglang-server
```

> 💡 처음 실행 시 Qwen3-8B 모델 다운로드로 약 16GB 다운로드가 필요해요!

## 0. GPU 확인 (제일 먼저 실행!)

In [23]:
import subprocess
result = subprocess.run(
    ['docker', 'compose', 'up', '-d'],
    capture_output=True,
    text=True,
    cwd=r'C:\Users\SSAFY\Desktop\ㄷㅇ\mine-tuning-model\docker'  # docker 폴더 절대경로
)
print(result.stdout)
print(result.stderr)


 Container sglang-server Creating 
 Container sglang-server Created 
 Container sglang-server Starting 
 Container sglang-server Started 



In [24]:
import requests
import time

print("⏳ SGLang 서버 연결 확인 중...")

try:
    r = requests.get("http://localhost:30000/health")
    if r.status_code == 200:
        print("SGLang 서버 정상 동작 중!")
except:
    print('연결 실패')

⏳ SGLang 서버 연결 확인 중...
SGLang 서버 정상 동작 중!


## 1. 필수 라이브러리 설치

> 💡 처음 한 번만 실행하면 돼요!

In [25]:
import subprocess
subprocess.run(['pip', 'install', 'requests', '-q'], check=True)
print("완료")

완료


In [26]:
import wandb

wandb.login()

print('✅ wandb 로그인 완료')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\SSAFY\_netrc.
wandb: Currently logged in as: ddorin789 (ddorin789-nothing) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ wandb 로그인 완료


## 2. 데이터 로드 및 Instruction 형식 변환

In [27]:
from datasets import load_dataset

# 데이터 로드
ds = load_dataset('ddorin/minecraft-question-answer-260k')
print(ds)

# Instruction 형식으로 변환
def format_instruction(example):
    return {
        'text': f"""<|im_start|>system
You are a helpful Minecraft expert assistant.
<|im_end|>
<|im_start|>user
{example['question']}
<|im_end|>
<|im_start|>assistant
{example['answer']}
<|im_end|>"""
    }

train_data = ds['train'].map(format_instruction, remove_columns=['question', 'answer', 'source'])
print(f'✅ 데이터 변환 완료: {len(train_data)}개')
print('\n샘플 확인:')
print(train_data[0]['text'])

c:\Users\SSAFY\Desktop\ㄷㅇ\mine-tuning-model\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'source'],
        num_rows: 268239
    })
})
✅ 데이터 변환 완료: 268239개

샘플 확인:
<|im_start|>system
You are a helpful Minecraft expert assistant.
<|im_end|>
<|im_start|>user
What types of leaves can be found naturally in jungle bushes in Minecraft?
<|im_end|>
<|im_start|>assistant
In Minecraft, jungle bushes can generate oak leaves in the Java Edition and jungle leaves in the Bedrock Edition.
<|im_end|>


## 3.추론 함수

In [ ]:
import requests
import json

SGLANG_URL = "http://localhost:30000/v1/chat/completions"

def chat(prompt, thinking=False):
    payload = {
        "model": "Qwen/Qwen3-8B",
        "messages": [
            {"role": "system", "content": "You are a helpful Minecraft expert assistant."},
            {"role": "user",   "content": prompt}
        ],
        "temperature": 0.6,
        "top_p": 0.95,
        "max_tokens": 512,
        "extra_body": {"separate_reasoning": thinking}
    }
    resp = requests.post(SGLANG_URL, json=payload)
    data = resp.json()

    if thinking:
        reasoning = data["choices"][0]["message"].get("reasoning_content", "")
        content   = data["choices"][0]["message"]["content"]
        print("🧠 [Thinking]\n", reasoning)
        print("\n💬 [Answer]\n", content)
    else:
        print(data["choices"][0]["message"]["content"])

print("✅ 추론 함수 준비 완료")

## 기본 추론 테스트

In [32]:
import requests
import json

SGLANG_URL = "http://localhost:30000/v1/chat/completions"

def chat(prompt, thinking=False):
    payload = {
        "model": "Qwen/Qwen3-8B",
        "messages": [
            {"role": "system", "content": "You are a helpful Minecraft expert assistant."},
            {"role": "user",   "content": prompt}
        ],
        "temperature": 0.6,
        "top_p": 0.95,
        "max_tokens": 2048,  # 늘림
        "extra_body": {
            "separate_reasoning": thinking,
            "chat_template_kwargs": {"enable_thinking": thinking}  # thinking 모드 명시적 제어
        }
    }
    resp = requests.post(SGLANG_URL, json=payload)
    data = resp.json()
    choice = data["choices"][0]["message"]

    if thinking:
        print("🧠 [Thinking]\n", choice.get("reasoning_content", ""))
        print("\n💬 [Answer]\n", choice.get("content", ""))
    else:
        print(choice.get("content", ""))

print("✅ 추론 함수 준비 완료")

✅ 추론 함수 준비 완료


In [33]:
# 기본 추론 테스트
chat("다이아몬드는 어디서 가장 잘 나와?")

다이아몬드는 **마인크래프트(Overworld)**에서 가장 잘 나오는 위치는 **Y=5~Y=12**의 **바위(Stone)**층입니다. 이 범위는 주로 **2차 지층**(Second Layer)이라고 불리며, 대부분의 플레이어들이 다이아몬드를 찾는 기본 영역입니다. 아래에 자세한 정보를 정리해 드릴게요:

---

### 🌍 **다이아몬드의 주요 출처**
1. **Y=5~Y=12의 바위층**  
   - 이 범위 내에서 **바위**, **진흙**, **갈색 진흙**, **모래**, **갈색 모래** 등의 블록 위에 다이아몬드가 생성됩니다.  
   - 다이아몬드는 **1~256**의 확률로 생성되며, 전체적으로 **1/256**의 확률로 나옵니다.  
   - **Nether**와 **End**에는 다이아몬드가 **존재하지 않습니다**.

2. **심층 탐사**  
   - Y=1~Y=5까지 가면 **돌(Stone)**이 아닌 **구리(Orichalcum)**, **철(Steel)**, **마그네슘(Magnesium)** 등이 주로 나오므로, 다이아몬드는 거의 없습니다.  
   - Y=12 이상으로 가면 **다이아몬드가 거의 나오지 않거나, 다른 자원(예: 루비, 블랙 다이아몬드)**이 주로 나타납니다.

---

### 🛠️ **다이아몬드 채굴 팁**
1. **적절한 도구 사용**  
   - **철 과외** 이상의 과외(예: **마가네슘 과외**, **다이아몬드 과외**)로 채굴해야 합니다.  
   - **Fortune** 엔CHANTMENT를 추가하면 **다이아몬드의 양이 증가**합니다.

2. **지형 활용**  
   - **동굴**, **지하수**, **사자들의 터널** 등에서 바위층을 탐색하세요.  
   - **서로 다른 지점에서 채굴**하면 다이아몬드를 더 쉽게 찾을 수 있습니다.

3. **확률 증가 방법**  
   - **다이아몬드 채굴기**를 사용하면 확률이 약 1.5배 증가합니다.  
   - **다이아몬드 채굴기**는 **마인크

In [34]:
# Thinking Test
chat("크리퍼를 안전하게 처치하는 방법 3가지 알려줘", thinking=True)

🧠 [Thinking]
 Okay, the user is asking for three safe ways to kill creepers in Minecraft. Let me start by recalling what I know about creepers. They're hostile mobs that explode, so the key is to avoid getting too close. First, maybe using ranged weapons like bows or arrows. That way, you can keep a distance. But wait, arrows can sometimes miss, so maybe using a bow with fire resistance potions would help. Also, maybe using a sword with fire resistance enchantments. But wait, the user wants safe methods, so maybe using potions to reduce damage is important.

Another idea is using a tnt minecart. Since creepers are attracted to explosions, placing tnt near them might work. But you need to be careful not to get too close. Maybe using a dispenser with fire charge or a potion of fire resistance. Also, using a shield could help block some damage if you have to get close. Oh, but the user asked for three methods. Let me think of three distinct approaches. 

First, using ranged attacks with b

# 셀 6 — 버전 확인

In [35]:
import requests
import json
import subprocess

# 서버 모델 정보
info = requests.get("http://localhost:30000/v1/models").json()
print("서버 모델 정보:")
print(json.dumps(info, indent=2, ensure_ascii=False))

# Docker 버전
result = subprocess.run(["docker", "--version"], capture_output=True, text=True)
print("\nDocker:", result.stdout.strip())

서버 모델 정보:
{
  "object": "list",
  "data": [
    {
      "id": "Qwen/Qwen3-8B",
      "object": "model",
      "created": 1779772344,
      "owned_by": "sglang",
      "root": "Qwen/Qwen3-8B",
      "parent": null,
      "max_model_len": 40960
    }
  ]
}

Docker: Docker version 29.2.1, build a5c7197
